# Day 14 — Machine Learning Fundamentals

**Task:** Understand the machine learning workflow.

This beginner notebook uses a small fictional customer dataset to predict whether a customer may leave a service. The goal is to practise a complete workflow, not to build a production-ready model. Run each cell from top to bottom.

**Dataset file:** `day14_customer_churn_sample.csv`


## 1. What is machine learning?

Machine learning (ML) allows a computer to learn patterns from examples and use those patterns to make predictions.

- **Supervised learning:** the training data includes a known answer, called a label. Example: predicting customer churn.
- **Unsupervised learning:** the data has no target label. The model looks for groups or patterns, such as grouping customers by behaviour.
- **Classification:** predicts a category, such as churn = Yes/No.
- **Regression:** predicts a number, such as next month's sales.

This notebook uses **supervised classification** because `Churn` is a known 0/1 label.


## 2. Load and understand the data

In [ ]:
import pandas as pd

df = pd.read_csv("day14_customer_churn_sample.csv")
print("Dataset shape:", df.shape)
display(df.head())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nTarget counts (0 = stays, 1 = churns):")
print(df["Churn"].value_counts())


## 3. Separate features and target

In [ ]:
# Features are the information used to make a prediction.
# Target is the answer the model learns to predict.
X = df[["TenureMonths", "MonthlyCharge", "SupportCalls", "ContractType"]]
y = df["Churn"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)


## 4. Split training, validation, and test data

- **Training data** teaches the model patterns.
- **Validation data** helps compare model choices while developing.
- **Test data** is held back until the end to estimate performance on unseen examples.

For this beginner baseline, we create a 70% training set, 15% validation set, and 15% test set. The validation set is used to review the baseline; the test set is saved for a final check.


In [ ]:
from sklearn.model_selection import train_test_split

# First hold back 30% of the data.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
# Split the remaining 30% equally into validation and test (15% each overall).
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))
print("Test rows:", len(X_test))


## 5. Build a simple baseline classification model

A baseline is a first, simple model that gives us a starting point. We use a `DummyClassifier` that always predicts the most common class. Then we compare it with a simple Logistic Regression model. The preprocessing step converts the text contract type into numbers and scales numeric columns.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error

numeric_features = ["TenureMonths", "MonthlyCharge", "SupportCalls"]
categorical_features = ["ContractType"]

preprocess = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

baseline = Pipeline([
    ("preprocess", preprocess),
    ("model", DummyClassifier(strategy="most_frequent"))
])
baseline.fit(X_train, y_train)
valid_baseline_pred = baseline.predict(X_valid)

model = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])
model.fit(X_train, y_train)
valid_pred = model.predict(X_valid)

def show_classification_metrics(name, actual, predicted):
    print(name)
    print(f"Accuracy : {accuracy_score(actual, predicted):.3f}")
    print(f"Precision: {precision_score(actual, predicted, zero_division=0):.3f}")
    print(f"Recall   : {recall_score(actual, predicted, zero_division=0):.3f}")
    print(f"F1-score : {f1_score(actual, predicted, zero_division=0):.3f}")
    print(f"MAE      : {mean_absolute_error(actual, predicted):.3f}")
    print()

show_classification_metrics("Baseline on validation data", y_valid, valid_baseline_pred)
show_classification_metrics("Logistic Regression on validation data", y_valid, valid_pred)


## 6. Understand the evaluation metrics

- **Accuracy:** the share of all predictions that are correct.
- **Precision:** among customers predicted to churn, how many actually churned.
- **Recall:** among customers who actually churned, how many the model found.
- **F1-score:** balances precision and recall in one score.
- **MAE (Mean Absolute Error):** average absolute difference between actual and predicted values. For this 0/1 classification exercise it can be calculated, but it is more commonly used for regression tasks. For classification, focus mainly on accuracy, precision, recall, and F1-score.

A model can have good accuracy and still miss many churn cases, especially if one class is much more common than the other. That is why several metrics are useful.


## 7. Final evaluation on the test set

In [ ]:
# After reviewing the validation results, check the chosen model once on held-out test data.
test_pred = model.predict(X_test)
show_classification_metrics("Logistic Regression on test data", y_test, test_pred)


## 8. Overfitting and underfitting

- **Overfitting** happens when a model learns the training examples too closely, including noise, and performs worse on new data.
- **Underfitting** happens when a model is too simple to capture important patterns, so it performs poorly even on training data.
- Compare training and validation results to look for a large performance gap. This notebook is a basic introduction; it does not claim to diagnose either issue conclusively.

## 9. What I learned

I learned the difference between supervised and unsupervised learning, classification and regression, and the purpose of training, validation, and test datasets. I trained a baseline classifier and a Logistic Regression model using Scikit-learn, and evaluated predictions with accuracy, precision, recall, F1-score, and MAE.

## Limitations

This is a small fictional dataset created for practice. Its patterns are artificial and do not represent real customers. Results can change with a different train/test split, and a real business model would need better data, careful validation, and checks for fairness and usefulness before being used.

## Conclusion

A machine-learning workflow includes understanding the problem, preparing the data, splitting it correctly, training a model, and evaluating predictions. A baseline gives a useful starting point, while multiple metrics provide a more complete picture than accuracy alone.
